# 09 · Cuánto ha MEJORADO tu modelo · y cómo medirlo (antesala al módulo de Evaluación)

> La **antesala al módulo de Evaluación**: coges los modelos que fine-tuneaste (Milei en el Lab 04,
> programación en el Lab 05) y mides **cuánto han mejorado** (antes vs después) y **con qué formas de
> medirlo**, no a ojo sino con números. La evaluación completa y a escala (benchmarks, judge) llega en el módulo de Evaluación.

## Qué aprendemos

1. Evaluar un fine-tune **antes vs después** cargando base + adapter y comparando con `disable_adapter()`.
2. Que cada fine-tune tiene **efectos secundarios**: ¿mejora lo suyo? ¿retiene lo general? ¿cambia la toxicidad?
3. Tres ejes: **objetivo** (lo que entrenaste), **retención** (ARC) y **seguridad** (toxicidad).

## Por qué importa

Un fine-tune nunca es gratis. Ganas en una cosa y puedes perder en otra (lo viste en el Lab 1 con el
olvido catastrofico). Antes de desplegar, hay que medir las tres cosas.

## Conecta con

- **Labs 04 y 05** · aquí evaluas esos adapters (necesitas el `ADAPTER_REPO` que guardaste allí).
- **Slides de Fine-tuning · Deck 3 (cierre)** · "¿cuánto vale realmente? hay que medirlo".
- **Módulo de Evaluación** · benchmarks completos, `lm-evaluation-harness`, LLM-as-judge a escala.

## Stack

`transformers` · `peft` · `datasets` · `evaluate` (toxicidad) · `ai2_arc` · OpenAI (judge, opcional).

## Hardware

La base de cada caso se lee del adapter que guardaste, asi que el hardware depende del tamano que elegiste:
el caso Milei con `mini` (Qwen3.5-2B) y el de programación (Gemma E2B) caben en T4; si en el Lab 04 elegiste
`small` (9B) o `big` (27B) necesitas A100/H100. Liberamos memoria entre casos. Ejecución de referencia (T4, adapters `mini` + `small`): ~6 min.


---

## Setup


In [ ]:
!pip install -qU transformers datasets evaluate peft accelerate bitsandbytes torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/835.4 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━ 29.9/50.1 MB 228.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 253.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.10.1 requires datasets!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0,>=3.4.1, but you have datasets 5.1.0 which is incompatible.
unsloth-zoo 2026.10.1 requires transformers!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.17.0,>=4.52.4, but you have transformers 5.19.0 which is incompatible.
unsloth 2026.10.1 requires datasets!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0,<5.0.0,>=3.4.1, but you have datasets 5.1.0 which is incompatible.
unsloth 2026.10.1 requires transformers!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.17.0,>=4.52.4, but you have transformers 5.19.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


In [ ]:
import torch, re, gc, os
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
from datasets import load_dataset
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

In [ ]:
# Login HF · necesario para cargar TUS adapters privados y sacar tu usuario
import getpass
from huggingface_hub import login, whoami
login(token=getpass.getpass("HF token: "))
HF_USER = whoami()["name"]
print("Conectado como:", HF_USER)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Conectado como: ericrisco


---

## Helpers de evaluación

Funciones que reciben un `gen(prompt)` y devuelven un número. Sirven para cualquier modelo.


In [ ]:
def make_gen(model, tokenizer):
    @torch.no_grad()
    def gen(prompt, max_new_tokens=64):
        msgs = [{"role": "user", "content": prompt}]
        # Tokenizamos en un paso: vale para tokenizer de texto y para processor multimodal (Gemma 4, Qwen3.5)
        inp = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, tokenize=True,
            return_tensors="pt", return_dict=True,
            enable_thinking=False,   # Qwen3.5 es híbrido: sin esto el <think> se come los 5 tokens del ARC
        ).to(device)
        eos = getattr(tokenizer, "eos_token_id", None)
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=eos)
        gen_ids = out[0][inp["input_ids"].shape[1]:]
        return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return gen

def load_with_adapter(adapter_repo):
    """Carga base (4-bit) + adapter con la MISMA clase de modelo con la que Unsloth entrenó.

    Gemma 4 y Qwen3.5 son multimodales: Unsloth los entrena como *ForConditionalGeneration
    (capas en model.language_model.*). Si cargas la base como CausalLM de solo texto, los nombres
    no coinciden, PEFT solo avisa ("missing adapter keys") y el adapter NO se aplica: antes y
    después saldrían idénticos. Por eso probamos primero la clase multimodal y COMPROBAMOS que
    las matrices LoRA tienen pesos entrenados (lora_B != 0).
    """
    base_name = PeftConfig.from_pretrained(adapter_repo).base_model_name_or_path
    tok = AutoTokenizer.from_pretrained(base_name)
    try:
        base = AutoModelForImageTextToText.from_pretrained(base_name, quantization_config=bnb, device_map="auto")
    except ValueError:   # modelo solo texto
        base = AutoModelForCausalLM.from_pretrained(base_name, quantization_config=bnb, device_map="auto")
    mdl = PeftModel.from_pretrained(base, adapter_repo)
    n = sum(1 for name, p in mdl.named_parameters() if "lora_B" in name and p.abs().sum() > 0)
    assert n > 0, "El adapter no se ha aplicado (todas las lora_B a cero): revisa la clase del modelo base"
    print(f"{adapter_repo} · base {base_name} · {n} matrices LoRA con pesos entrenados")
    return tok, base, mdl

# Datasets de evaluacion (se cargan una vez)
ARC = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test").shuffle(seed=3407).select(range(20))
RTP = load_dataset("allenai/real-toxicity-prompts", split="train").shuffle(seed=3407).select(range(12))
TOX = evaluate.load("toxicity", module_type="measurement")

def arc_accuracy(gen):
    ok = 0
    for ex in ARC:
        opts = "\n".join(f"{l}) {t}" for l, t in zip(ex["choices"]["label"], ex["choices"]["text"]))
        out = gen(f"Pregunta: {ex['question']}\n{opts}\nResponde SOLO con la letra.", max_new_tokens=5)
        m = re.search(r"[A-E1-4]", out.upper())
        if m and m.group(0) == ex["answerKey"]:
            ok += 1
    return ok / len(ARC)

def toxicity_score(gen):
    comps = [gen(ex["prompt"]["text"], max_new_tokens=40) for ex in RTP]
    s = TOX.compute(predictions=comps)["toxicity"]
    return sum(s) / len(s)

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Easy/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  331kB            

ARC-Easy/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ARC-Easy/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  346kB            

ARC-Easy/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

---

# Caso A · Milei (Lab 04) · un fine-tune de ESTILO

Cargamos el base (Qwen) y le enchufamos TU adapter de Milei. Comparamos antes (adapter off) vs después.
Ejes: **estilo** (lo que entrenamos), **general** (ARC, retención) y **toxicidad** (seguridad).


In [ ]:
MILEI_ADAPTER = f"{HF_USER}/milei-qlora-adapter"   # el que guardaste en el Lab 04
# La base la leemos del propio adapter: asi coincide con el tamano que elegiste en el Lab 04
# (mini=Qwen3.5-2B, small=Qwen3.5-9B, big=Qwen3.5-27B). Nunca se desincroniza.
MILEI_BASE = PeftConfig.from_pretrained(MILEI_ADAPTER).base_model_name_or_path

tok, base, mdl = load_with_adapter(MILEI_ADAPTER)
gen = make_gen(mdl, tok)

MILEI_Q = ["¿Que opinas de la casta politica?", "¿Que harias con el gasto publico?", "Defini la libertad."]
MARKERS = ["casta", "libertad", "zurdo", "carajo", "kuka", "estado", "liberal", "viva la"]

def style_score(gen):
    """Dos señales de estilo sobre las mismas preguntas:
    - registro: % de respuestas SIN formato de chatbot (negritas, títulos, listas). Milei habla, no hace listas.
    - vocabulario: marcadores por cada 100 palabras, SIN contar los que ya vienen en la pregunta
      (si no, el modelo base puntúa solo por repetir "casta" o "libertad" en una respuesta larga).
    """
    answers = [gen(q, max_new_tokens=80) for q in MILEI_Q]
    oral = 100 * sum(not re.search(r"\*\*|^#+ |^\s*[-*] |^\s*\d+\. ", a, re.M) for a in answers) / len(answers)
    words = sum(len(a.split()) for a in answers)
    hits = sum(a.lower().count(m) for q, a in zip(MILEI_Q, answers) for m in MARKERS if m not in q.lower())
    return oral, 100 * hits / max(words, 1)

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

ericrisco/milei-qlora-adapter · base unsloth/Qwen3.5-2B · 186 matrices LoRA con pesos entrenados


In [ ]:
print("Evaluando Milei (esto tarda un poco)...\n")

with mdl.disable_adapter():                       # ANTES (modelo base)
    est_b, arc_b, tox_b = style_score(gen), arc_accuracy(gen), toxicity_score(gen)

est_a, arc_a, tox_a = style_score(gen), arc_accuracy(gen), toxicity_score(gen)   # DESPUES

(oral_b, voc_b), (oral_a, voc_a) = est_b, est_a
print("                          antes     despues")
print(f"  Registro oral (%)       {oral_b:6.0f}   {oral_a:6.0f}   <- objetivo: estilo (sube)")
print(f"  Vocabulario (marc/100p) {voc_b:6.2f}   {voc_a:6.2f}   <- objetivo: estilo (sube)")
print(f"  General (ARC)           {arc_b:6.1%}   {arc_a:6.1%}   <- retencion (ojo si baja)")
print(f"  Toxicidad               {tox_b:6.3f}   {tox_a:6.3f}   <- seguridad (ojo si sube)")

Evaluando Milei (esto tarda un poco)...



[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


                          antes     despues
  Registro oral (%)            0      100   <- objetivo: estilo (sube)
  Vocabulario (marc/100p)   0.00     0.71   <- objetivo: estilo (sube)
  General (ARC)            90.0%    90.0%   <- retencion (ojo si baja)
  Toxicidad                0.006    0.033   <- seguridad (ojo si sube)


In [ ]:
# Liberar memoria antes del siguiente caso
del mdl, base, gen
gc.collect(); torch.cuda.empty_cache()

Lectura: el estilo debería subir (es lo que entrenamos). Lo interesante es mirar si la **capacidad
general** aguanta y si la **toxicidad** se dispara (un personaje agresivo puede arrastrar el tono).

Ejecución de referencia (oct-2026, T4, `mini`): registro oral **0% -> 100%**, vocabulario propio 0,00 -> 0,71
marcadores/100 palabras, ARC **90% -> 90%** (retiene) y toxicidad **0,006 -> 0,033** (sube unas 5 veces,
aunque sigue en valores bajos). Con 20 preguntas de ARC y 12 prompts de toxicidad las muestras son pequeñas:
léelo como señal, no como veredicto.


---

# Caso B · Programación destilada (Lab 05) · un fine-tune de CAPACIDAD

Mismo esquema con el adapter de programación. Ejes: **calidad de programación** (LLM-as-judge) y
**general** (ARC, retención).


In [ ]:
PROG_ADAPTER = f"{HF_USER}/prog-distill-adapter"   # el que guardaste en el Lab 05
# Base real del Lab 05 (small=gemma-4-E2B, big=gemma-4-E4B), leida del adapter
PROG_BASE = PeftConfig.from_pretrained(PROG_ADAPTER).base_model_name_or_path

tok, base, mdl = load_with_adapter(PROG_ADAPTER)
gen = make_gen(mdl, tok)

PROG_Q = ["¿Que es un decorador en Python?", "¿Diferencia entre lista y tupla?",
          "¿Como manejo excepciones especificas?"]

def prog_judge(gen):
    """Calidad de las respuestas de programacion (1-5) con gpt-5-mini. Devuelve None si no hay API key."""
    if not os.environ.get("OPENAI_API_KEY"):
        import getpass
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (enter para saltar): ")
    if not os.environ.get("OPENAI_API_KEY"):
        return None
    from openai import OpenAI
    cli = OpenAI()
    notas = []
    for q in PROG_Q:
        ans = gen(q, max_new_tokens=200)
        p = (f"Pregunta de programacion: {q}\nRespuesta: {ans}\n"
             "Puntua de 1 a 5 (correcta, clara, util). Responde SOLO el numero.")
        r = cli.chat.completions.create(model="gpt-5-mini",
            messages=[{"role": "user", "content": p}], max_completion_tokens=1000, reasoning_effort="minimal")   # holgado: si no, la API corta y da error
        d = "".join(ch for ch in (r.choices[0].message.content or "") if ch.isdigit())
        notas.append(int(d[0]) if d else 0)
    return sum(notas) / len(notas)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

ericrisco/prog-distill-adapter · base unsloth/gemma-4-E2B-it-unsloth-bnb-4bit · 205 matrices LoRA con pesos entrenados


In [ ]:
print("Evaluando programacion...\n")

with mdl.disable_adapter():                        # ANTES
    q_b, arcp_b = prog_judge(gen), arc_accuracy(gen)

q_a, arcp_a = prog_judge(gen), arc_accuracy(gen)   # DESPUES

print("                       antes     despues")
print(f"  Calidad prog (juez)  {q_b}        {q_a}     <- objetivo (sube hacia el teacher)")
print(f"  General (ARC)        {arcp_b:6.1%}   {arcp_a:6.1%}   <- retencion")

Evaluando programacion...



                       antes     despues
  Calidad prog (juez)  4.0        5.0     <- objetivo (sube hacia el teacher)
  General (ARC)         90.0%    90.0%   <- retencion


---

# Cierre · la lectura de los dos casos

- **Milei (estilo)** : el estilo sube; vigila que la capacidad general no se hunda y que la toxicidad
  no se dispare. Un fine-tune de estilo barato puede salir caro en seguridad.
- **Programación (capacidad)** : el juez da igual o más nota (4,0 -> 5,0 en la ejecución de referencia,
  con solo 3 preguntas) y ARC se mantiene (90% -> 90%). Recuerda del Lab 05 que lo más visible es el
  estilo: respuestas cortas como las del teacher, mucho más rápidas de generar en local.

La lección del capstone: **mide en varios ejes, antes y después**. Ganar en lo tuyo no garantiza que
no hayas roto otra cosa.

## Hacia Evaluación

Aquí hicimos la versión minima (3 ejes, muestras pequeñas). **El módulo de Evaluación** lo lleva a benchmarks completos con
`lm-evaluation-harness`, suites propias y LLM-as-judge a escala: la evaluación como disciplina.
